In [7]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [8]:
load_dotenv()
model=ChatGoogleGenerativeAI(
    model='gemini-3.5-flash-lite',
    google_api_key=os.getenv("GEMINI_API_KEY_3")
)

In [9]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str

In [10]:
def genJoke(state:JokeState):
    prompt=f"Generate joke on the topic {state['topic']}"
    res=model.invoke(prompt).content[0]['text']
    return {'joke':res}

In [11]:
def genExplain(state:JokeState):
    prompt=f"Explain the joke: {state['joke']}"
    res=model.invoke(prompt).content[0]['text']
    return {'explaination':res}
    

In [13]:
graph=StateGraph(JokeState)

graph.add_node('genJoke',genJoke)
graph.add_node('genExplain',genExplain)

graph.add_edge(START,'genJoke')
graph.add_edge('genJoke','genExplain')
graph.add_edge('genExplain',END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [14]:
config1={'configurable':{'thread_id':'1'}}
workflow.invoke({'topic':'burger'},config=config1)

{'topic': 'burger',
 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!',
 'explaination': 'This joke relies on a **double entendre** (a word or phrase with two meanings) based on the word "buns."\n\n1. **Food meaning:** A hamburger bun is the bread that holds the burger together. \n2. **Slang meaning:** "Buns" is also a common slang term for buttock muscles. \n\nGoing to the gym is something people do to tone and improve their muscles. Therefore, the joke plays on the idea of a hamburger wanting to improve its physical appearance by working out its "buns" (both its bread and its rear end).'}

In [15]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!', 'explaination': 'This joke relies on a **double entendre** (a word or phrase with two meanings) based on the word "buns."\n\n1. **Food meaning:** A hamburger bun is the bread that holds the burger together. \n2. **Slang meaning:** "Buns" is also a common slang term for buttock muscles. \n\nGoing to the gym is something people do to tone and improve their muscles. Therefore, the joke plays on the idea of a hamburger wanting to improve its physical appearance by working out its "buns" (both its bread and its rear end).'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-d008-64cd-8002-62fdfce8a602'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-15T11:31:39.587477+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-8ab1-6b4c-8001-6230193343ee'}}, t

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!', 'explaination': 'This joke relies on a **double entendre** (a word or phrase with two meanings) based on the word "buns."\n\n1. **Food meaning:** A hamburger bun is the bread that holds the burger together. \n2. **Slang meaning:** "Buns" is also a common slang term for buttock muscles. \n\nGoing to the gym is something people do to tone and improve their muscles. Therefore, the joke plays on the idea of a hamburger wanting to improve its physical appearance by working out its "buns" (both its bread and its rear end).'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-d008-64cd-8002-62fdfce8a602'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-15T11:31:39.587477+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-8ab1-6b4c-8001-6230193343ee'}}, 

In [19]:
config2={'configurable':{'thread_id':'2'}}
workflow.invoke({'topic':'AI'},config=config2)

{'topic': 'AI',
 'joke': "Why did the AI go to art school?\n\nBecause it wanted to learn how to draw a crowd, but it just kept tracing everyone else's homework!",
 'explaination': 'This joke plays on two main concepts: how generative AI creates art, and common experiences from school. Here is the breakdown:\n\n1. **"Draw a crowd":** \n   * *In everyday English*, this is an idiom that means to attract a large group of people (like a performer on a street corner).\n   * *In the context of art*, it literally means to draw a picture of a group of people. \n\n2. **"Tracing everyone else\'s homework":** \n   * This is the punchline that connects the AI to the "art school" setting. \n   * **In school**, tracing someone else\'s homework is a form of cheating where you copy someone else\'s work instead of doing it yourself.\n   * **In technology**, this is a reference to how many current AI image generators work. They are trained by "learning" from millions of existing images created by human a

In [20]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'AI', 'joke': "Why did the AI go to art school?\n\nBecause it wanted to learn how to draw a crowd, but it just kept tracing everyone else's homework!", 'explaination': 'This joke plays on two main concepts: how generative AI creates art, and common experiences from school. Here is the breakdown:\n\n1. **"Draw a crowd":** \n   * *In everyday English*, this is an idiom that means to attract a large group of people (like a performer on a street corner).\n   * *In the context of art*, it literally means to draw a picture of a group of people. \n\n2. **"Tracing everyone else\'s homework":** \n   * This is the punchline that connects the AI to the "art school" setting. \n   * **In school**, tracing someone else\'s homework is a form of cheating where you copy someone else\'s work instead of doing it yourself.\n   * **In technology**, this is a reference to how many current AI image generators work. They are trained by "learning" from millions of existing images

In [22]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'AI', 'joke': "Why did the AI go to art school?\n\nBecause it wanted to learn how to draw a crowd, but it just kept tracing everyone else's homework!", 'explaination': 'This joke plays on two main concepts: how generative AI creates art, and common experiences from school. Here is the breakdown:\n\n1. **"Draw a crowd":** \n   * *In everyday English*, this is an idiom that means to attract a large group of people (like a performer on a street corner).\n   * *In the context of art*, it literally means to draw a picture of a group of people. \n\n2. **"Tracing everyone else\'s homework":** \n   * This is the punchline that connects the AI to the "art school" setting. \n   * **In school**, tracing someone else\'s homework is a form of cheating where you copy someone else\'s work instead of doing it yourself.\n   * **In technology**, this is a reference to how many current AI image generators work. They are trained by "learning" from millions of existing image

## TimeTravel

In [23]:
workflow.get_state({'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-47ca-66d1-8000-237a3c1d8615'}})

StateSnapshot(values={'topic': 'burger'}, next=('genJoke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-47ca-66d1-8000-237a3c1d8615'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-15T11:31:25.301511+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-47c7-61d9-bfff-aed04edd894d'}}, tasks=(PregelTask(id='e4a3a63d-a11e-bfa6-4027-9432eb22c872', name='genJoke', path=('__pregel_pull', 'genJoke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the burger go to the gym?\n\nTo get better buns!'}),), interrupts=())

In [24]:
workflow.invoke(None,{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-47ca-66d1-8000-237a3c1d8615'}})

{'topic': 'burger',
 'joke': 'Why did the burger go to the gym?\n\nTo get a little *bolder*! \n\n*(Okay, actually it was just trying to work on its buns.)*',
 'explaination': 'This joke works on two levels of wordplay (puns) related to food and fitness:\n\n1. **The Setup Punchline ("To get a little *bolder*!"):** \n   * A hamburger is made of beef, and a stronger, more flavorful taste is often described as "bolder" (as in a bold flavor). \n   * However, phonetically, it sounds like "bull-der" (like a bull, since beef comes from cattle), or it plays on the idea of a burger trying to toughen up. \n\n2. **The "Actually" Punchline ("...work on its buns"):** \n   * This is the real punchline of the joke. In fitness, people go to the gym to work on their "buns" (glute muscles). \n   * For a hamburger, its "buns" are the pieces of bread holding it together. \n\nIn short: A burger goes to the gym for the same reason humans do—to improve its body (its buns)!'}

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get a little *bolder*! \n\n*(Okay, actually it was just trying to work on its buns.)*', 'explaination': 'This joke works on two levels of wordplay (puns) related to food and fitness:\n\n1. **The Setup Punchline ("To get a little *bolder*!"):** \n   * A hamburger is made of beef, and a stronger, more flavorful taste is often described as "bolder" (as in a bold flavor). \n   * However, phonetically, it sounds like "bull-der" (like a bull, since beef comes from cattle), or it plays on the idea of a burger trying to toughen up. \n\n2. **The "Actually" Punchline ("...work on its buns"):** \n   * This is the real punchline of the joke. In fitness, people go to the gym to work on their "buns" (glute muscles). \n   * For a hamburger, its "buns" are the pieces of bread holding it together. \n\nIn short: A burger goes to the gym for the same reason humans do—to improve its body (its buns)!'}, next=(), conf

## Update State

In [30]:
# Fixed with proper single curly braces for the dictionary
workflow.update_state(
    {'configurable': {'thread_id': '1', 'checkpoint_ns': '','checkpoint_id': '1f1c88bf-47ca-66d1-8000-237a3c1d8615'}},
    {'topic': 'Momo'}
)

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1c892c-aa6b-65f9-8001-ce1c6de45e17'}}

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Momo'}, next=('genJoke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c892c-aa6b-65f9-8001-ce1c6de45e17'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-10-15T12:20:21.589924+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c88bf-47ca-66d1-8000-237a3c1d8615'}}, tasks=(PregelTask(id='4cfee0f5-c68c-071f-8c4b-59f14db3be2d', name='genJoke', path=('__pregel_pull', 'genJoke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get a little *bolder*! \n\n*(Okay, actually it was just trying to work on its buns.)*', 'explaination': 'This joke works on two levels of wordplay (puns) related to food and fitness:\n\n1. **The Setup Punchline ("To get a little *bolder*!"):** \n   * A hamburger is made of beef, and a stronger, more flavorfu

In [34]:
workflow.invoke(None,{'configurable': {'thread_id': '1', 'checkpoint_ns': '','checkpoint_id': '1f1c892c-aa6b-65f9-8001-ce1c6de45e17'}})

{'topic': 'Momo',
 'joke': 'Why did the momo go to therapy?\n\nBecause it had too many mixed feelings bottled up inside, and its skin was about to burst from the pressure!',
 'explaination': 'This joke is a clever play on words that combines human emotions with the physical characteristics of making and eating a momo (a type of South Asian dumpling). \n\nHere is the breakdown of why it\'s funny:\n\n1. **"Mixed feelings bottled up inside":** Humans go to therapy when they have unresolved emotional issues or "mixed feelings" that they keep hidden ("bottled up"). In a literal sense, a momo is filled with a *mixed* blend of ingredients (like minced meat, vegetables, and spices) packed tightly *inside* the dough.\n2. **"Skin was about to burst from the pressure":** In therapy, "pressure" refers to emotional stress. But for a momo, "skin" refers to the dough wrapper, and "pressure" refers to the steam and expanding heat when it is being cooked. If you stuff a momo too full or cook it too int

In [35]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Momo', 'joke': 'Why did the momo go to therapy?\n\nBecause it had too many mixed feelings bottled up inside, and its skin was about to burst from the pressure!', 'explaination': 'This joke is a clever play on words that combines human emotions with the physical characteristics of making and eating a momo (a type of South Asian dumpling). \n\nHere is the breakdown of why it\'s funny:\n\n1. **"Mixed feelings bottled up inside":** Humans go to therapy when they have unresolved emotional issues or "mixed feelings" that they keep hidden ("bottled up"). In a literal sense, a momo is filled with a *mixed* blend of ingredients (like minced meat, vegetables, and spices) packed tightly *inside* the dough.\n2. **"Skin was about to burst from the pressure":** In therapy, "pressure" refers to emotional stress. But for a momo, "skin" refers to the dough wrapper, and "pressure" refers to the steam and expanding heat when it is being cooked. If you stuff a momo too ful